In [1]:
import pandas as pd
import numpy as np
import itertools
import time
from dateutil.relativedelta import relativedelta
from scipy.optimize import minimize
from scipy.special import gammaln

In [2]:
df = pd.read_csv("results.csv", parse_dates=["date"])
print(f"Total matches in dataset: {len(df)}")
print(f"Date range: {df['date'].min().date()} to {df['date'].max().date()}")
print(f"Unique tournaments: {df['tournament'].nunique()}")



Total matches in dataset: 49071
Date range: 1872-11-30 to 2026-01-26
Unique tournaments: 191


In [3]:
df.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0,0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4,2,Friendly,London,England,False
2,1874-03-07,Scotland,England,2,1,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2,2,Friendly,London,England,False
4,1876-03-04,Scotland,England,3,0,Friendly,Glasgow,Scotland,False


 ### 3. Tournament Mapping and Filtering

 4 categories (clean international only):
 - `friendly` (0) — international friendlies
 - `qualifier` (1) — official qualifying campaigns
 - `confederation` (2) — 6 major continental championships only
 - `worldcup` (3) — FIFA World Cup finals

In [4]:
tournament_mapping = {
    # Friendly
    "Friendly": "friendly",

    # World Cup finals
    "FIFA World Cup": "worldcup",

    # Qualifiers
    "FIFA World Cup qualification": "qualifier",
    "AFC Asian Cup qualification": "qualifier",
    "African Cup of Nations qualification": "qualifier",
    "UEFA Euro qualification": "qualifier",
    "Copa América qualification": "qualifier",
    "Gold Cup qualification": "qualifier",
    "Oceania Nations Cup qualification": "qualifier",

    # Confederation — 6 major continental championships only
    "AFC Asian Cup": "confederation",
    "African Cup of Nations": "confederation",
    "UEFA Euro": "confederation",
    "Copa América": "confederation",
    "Gold Cup": "confederation",
    "Oceania Nations Cup": "confederation",
}

category_to_code = {
    "friendly": 0,
    "qualifier": 1,
    "confederation": 2,
    "worldcup": 3,
}
 

In [5]:
# Filter to last 8 years
last_date = df["date"].max()
start_date = last_date - relativedelta(years=8)
df_8y = df[df["date"] >= start_date].copy().reset_index(drop=True)
print(f"8-year window: {start_date.date()} to {last_date.date()}")
print(f"Total matches in 8-year window: {len(df_8y)}")

8-year window: 2018-01-26 to 2026-01-26
Total matches in 8-year window: 7800


In [6]:
# Apply mapping
df_8y["match_type"] = df_8y["tournament"].map(tournament_mapping)

In [7]:
dropped = df_8y[df_8y["match_type"].isna()]
kept = df_8y[df_8y["match_type"].notna()]

In [8]:
print(f"\nKept: {len(kept)} matches")
print(f"Dropped: {len(dropped)} matches\n")


Kept: 5789 matches
Dropped: 2011 matches



In [9]:
print("Dropped tournaments:")
for t, count in dropped["tournament"].value_counts().sort_values(ascending=False).items():
    print(f"    - {t} ({count} matches)")

Dropped tournaments:
    - UEFA Nations League (658 matches)
    - CONCACAF Nations League (422 matches)
    - COSAFA Cup (112 matches)
    - AFF Championship (78 matches)
    - CONCACAF Nations League qualification (68 matches)
    - Arab Cup (63 matches)
    - Pacific Games (46 matches)
    - CONIFA World Football Cup (45 matches)
    - Gulf Cup (45 matches)
    - Island Games (44 matches)
    - SAFF Cup (38 matches)
    - ASEAN Championship (26 matches)
    - EAFF Championship (24 matches)
    - FIFA Series (24 matches)
    - Indian Ocean Island Games (23 matches)
    - MSG Prime Minister's Cup (23 matches)
    - CONIFA European Football Cup (20 matches)
    - Inter Games (19 matches)
    - CONCACAF Series (19 matches)
    - Intercontinental Cup (17 matches)
    - WAFF Championship (17 matches)
    - King's Cup (16 matches)
    - Baltic Cup (14 matches)
    - Kirin Challenge Cup (14 matches)
    - Arab Cup qualification (13 matches)
    - CAFA Nations Cup (10 matches)
    - Muratti 

In [10]:
df_8y = kept.copy().reset_index(drop=True)
df_8y["category"] = df_8y["match_type"].map(category_to_code)

In [11]:
print("\nCategory breakdown:")
for match_type, code in category_to_code.items():
    matches = df_8y[df_8y["category"] == code]
    tournaments = sorted(matches["tournament"].unique())
    print(f"\n  [{code}] {match_type}: {len(matches)} matches")
    for t in tournaments:
        c = (matches["tournament"] == t).sum()
        print(f"       - {t} ({c})")


Category breakdown:

  [0] friendly: 2059 matches
       - Friendly (2059)

  [1] qualifier: 2967 matches
       - AFC Asian Cup qualification (118)
       - African Cup of Nations qualification (560)
       - Copa América qualification (2)
       - FIFA World Cup qualification (1751)
       - Gold Cup qualification (32)
       - Oceania Nations Cup qualification (3)
       - UEFA Euro qualification (501)

  [2] confederation: 635 matches
       - AFC Asian Cup (102)
       - African Cup of Nations (208)
       - Copa América (86)
       - Gold Cup (124)
       - Oceania Nations Cup (13)
       - UEFA Euro (102)

  [3] worldcup: 128 matches
       - FIFA World Cup (128)


In [12]:
print(f"\nTotal matches for modeling: {len(df_8y)}")


Total matches for modeling: 5789


### 4. Precompute All Constants

 - P(X=x, Y=y) = P(X=x) × P(Y=y)
 - P(X=x) = e^(-μ_home) × μ_home^x / x!
 - No k-loop needed for training (each goal count is independent)
 - Prediction grid still uses k-loop-free independent product

In [13]:
# --- Global log-factorial table ---
MAX_GOALS_OBS = int(max(df_8y["home_score"].max(), df_8y["away_score"].max())) + 1
MAX_GOALS_PRED = 10
MAX_GOALS = max(MAX_GOALS_OBS, MAX_GOALS_PRED) + 1
LOG_FACT = gammaln(np.arange(MAX_GOALS + 1) + 1)

In [14]:
# --- Prediction grid constants ---
PRED_GOALS = np.arange(MAX_GOALS_PRED + 1)
PRED_XX, PRED_YY = np.meshgrid(PRED_GOALS, PRED_GOALS, indexing="ij")
PRED_X_FLAT = PRED_XX.ravel()
PRED_Y_FLAT = PRED_YY.ravel()
PRED_N = len(PRED_X_FLAT)
PRED_SHAPE = (MAX_GOALS_PRED + 1, MAX_GOALS_PRED + 1)

In [15]:
# Precompute log-factorials for the prediction grid
PRED_LOG_FACT_X = LOG_FACT[PRED_X_FLAT]
PRED_LOG_FACT_Y = LOG_FACT[PRED_Y_FLAT]

In [16]:
# --- Outcome classification masks ---
MASK_HOME_WIN = np.tril(np.ones(PRED_SHAPE), -1).astype(bool)
MASK_DRAW = np.eye(MAX_GOALS_PRED + 1).astype(bool)
MASK_AWAY_WIN = np.triu(np.ones(PRED_SHAPE), 1).astype(bool)

In [17]:
print(f"Log-factorial table size: {len(LOG_FACT)}")
print(f"Prediction grid: {PRED_SHAPE[0]}x{PRED_SHAPE[1]} = {PRED_N} pairs")

Log-factorial table size: 18
Prediction grid: 11x11 = 121 pairs


In [18]:
# --- Category counts (for reference) ---
print("\nCategory counts:")
print(f"  {'Category':<16} {'Count':<10} {'% of total'}")
print(f"  {'─' * 40}")
total_matches_all = len(df_8y)
for name, code in category_to_code.items():
    count = int((df_8y["category"] == code).sum())
    pct = count / total_matches_all * 100
    print(f"  {name:<16} {count:<10} {pct:.1f}%")
print(f"  {'─' * 40}")
print(f"  {'Total':<16} {total_matches_all:<10}")


Category counts:
  Category         Count      % of total
  ────────────────────────────────────────
  friendly         2059       35.6%
  qualifier        2967       51.3%
  confederation    635        11.0%
  worldcup         128        2.2%
  ────────────────────────────────────────
  Total            5789      


### 5. Precompute Window Caches

In [19]:
def precompute_window(train_df, test_df, halfperiod=3 * 365.25):
    """
    Precompute all constants for a single rolling window.
    Called once per window, reused across all weight combinations.

    Independent Poisson — no k-loop needed for training.
    Log-likelihood decomposes into separate home and away terms.
    """
    # --- Training cache ---
    teams = sorted(set(train_df["home_team"]).union(set(train_df["away_team"])))
    team_to_idx = {t: i for i, t in enumerate(teams)}
    K = len(teams)
    n = len(train_df)

    h_indices = np.array([team_to_idx[t] for t in train_df["home_team"].values])
    a_indices = np.array([team_to_idx[t] for t in train_df["away_team"].values])
    home_scores = train_df["home_score"].values.astype(int)
    away_scores = train_df["away_score"].values.astype(int)
    not_neutral = ~train_df["neutral"].values.astype(bool)
    categories = train_df["category"].values.astype(int)

    # Temporal decay weights
    end_date = train_df["date"].max()
    days_ago = (end_date - train_df["date"]).dt.days.values
    w_time = 0.5 ** (days_ago / halfperiod)

    mean_goals = max(np.mean(np.concatenate([home_scores, away_scores])), 0.5)

    # Precompute log-factorials for observed scores (no k-loop needed)
    log_fact_home = LOG_FACT[home_scores]
    log_fact_away = LOG_FACT[away_scores]

    train_cache = {
        "teams": teams,
        "team_to_idx": team_to_idx,
        "K": K,
        "n": n,
        "h_indices": h_indices,
        "a_indices": a_indices,
        "home_scores": home_scores,
        "away_scores": away_scores,
        "not_neutral": not_neutral,
        "categories": categories,
        "w_time": w_time,
        "log_mean_goals": np.log(mean_goals),
        "log_fact_home": log_fact_home,
        "log_fact_away": log_fact_away,
    }

    # --- Test cache ---
    t_h_idx = []
    t_a_idx = []
    t_hs = []
    t_as = []
    t_nn = []

    for _, row in test_df.iterrows():
        ht = row["home_team"]
        at = row["away_team"]
        if ht in team_to_idx and at in team_to_idx:
            t_h_idx.append(team_to_idx[ht])
            t_a_idx.append(team_to_idx[at])
            t_hs.append(int(row["home_score"]))
            t_as.append(int(row["away_score"]))
            t_nn.append(not bool(row["neutral"]))

    n_valid = len(t_h_idx)

    # Precompute actual cumulative outcome vectors for RPS
    t_hs_arr = np.array(t_hs, dtype=int)
    t_as_arr = np.array(t_as, dtype=int)
    cum_actual = np.zeros((n_valid, 2))
    for i in range(n_valid):
        if t_hs_arr[i] > t_as_arr[i]:
            cum_actual[i] = [1.0, 1.0]
        elif t_hs_arr[i] == t_as_arr[i]:
            cum_actual[i] = [0.0, 1.0]
        else:
            cum_actual[i] = [0.0, 0.0]

    test_cache = {
        "h_indices": np.array(t_h_idx, dtype=int),
        "a_indices": np.array(t_a_idx, dtype=int),
        "not_neutral": np.array(t_nn, dtype=bool),
        "n_valid": n_valid,
        "cum_actual": cum_actual,
    }

    return train_cache, test_cache


In [20]:
def get_rolling_windows(df):
    first_date = df["date"].min()
    last_date = df["date"].max()
    train_start = first_date.replace(day=1)
    windows = []
    while True:
        test_start = train_start + relativedelta(months=11)
        test_end = test_start + relativedelta(months=1)
        if test_end > last_date + pd.Timedelta(days=1):
            break
        train_mask = (df["date"] >= train_start) & (df["date"] < test_start)
        test_mask = (df["date"] >= test_start) & (df["date"] < test_end)
        train_df = df[train_mask]
        test_df = df[test_mask]
        if len(train_df) > 0 and len(test_df) > 0:
            info = {
                "train_start": train_start,
                "train_end": test_start,
                "test_start": test_start,
                "test_end": test_end,
                "train_matches": len(train_df),
                "test_matches": len(test_df),
            }
            windows.append((train_df.copy(), test_df.copy(), info))
        train_start += relativedelta(months=1)
    return windows

In [21]:

print("Precomputing window caches...")
precompute_start = time.time()

Precomputing window caches...


In [22]:
raw_windows = get_rolling_windows(df_8y)
window_caches = []
for train_df, test_df, info in raw_windows:
    train_cache, test_cache = precompute_window(train_df, test_df)
    window_caches.append((train_cache, test_cache, info))

In [23]:
precompute_time = time.time() - precompute_start
print(f"Precomputed {len(window_caches)} windows in {precompute_time:.1f}s\n")

Precomputed 73 windows in 0.7s



In [24]:
print("First 5 windows:")
for i, (_, _, info) in enumerate(window_caches[:5]):
    print(f"  Window {i+1}: Train {info['train_start'].strftime('%Y-%m')} to "
          f"{info['train_end'].strftime('%Y-%m')} ({info['train_matches']} matches) → "
          f"Test {info['test_start'].strftime('%Y-%m')} ({info['test_matches']} matches)")

First 5 windows:
  Window 1: Train 2018-01 to 2018-12 (571 matches) → Test 2018-12 (24 matches)
  Window 2: Train 2018-02 to 2019-01 (589 matches) → Test 2019-01 (56 matches)
  Window 3: Train 2018-03 to 2019-02 (642 matches) → Test 2019-02 (5 matches)
  Window 4: Train 2018-04 to 2019-03 (510 matches) → Test 2019-03 (121 matches)
  Window 5: Train 2018-06 to 2019-05 (586 matches) → Test 2019-05 (7 matches)


In [25]:
print(f"\nLast window:")
_, _, info = window_caches[-1]
print(f"  Window {len(window_caches)}: Train {info['train_start'].strftime('%Y-%m')} to "
      f"{info['train_end'].strftime('%Y-%m')} ({info['train_matches']} matches) → "
      f"Test {info['test_start'].strftime('%Y-%m')} ({info['test_matches']} matches)")


Last window:
  Window 73: Train 2025-01 to 2025-12 (816 matches) → Test 2025-12 (36 matches)


### 6. Independent Poisson Model

Key difference from bivariate Poisson:
 - No correlation parameter λ₃
 - Home and away goals are independent
 - log P(X=x, Y=y) = log P(X=x) + log P(Y=y)
 - P(X=x) = e^(-μ) × μ^x / x!  →  log P = -μ + x×log(μ) - log(x!)
 - Only 2 global params: constant + home_advantage (vs 3 in bivariate)
 - Faster fitting, simpler prediction

In [26]:
def ip_logpmf(mu_home, mu_away, home_scores, away_scores, log_fact_home, log_fact_away):
    """
    Vectorized independent Poisson log-PMF.

    log P(X=x, Y=y) = log P(X=x) + log P(Y=y)
                     = (-μ_h + x·log(μ_h) - log(x!)) + (-μ_a + y·log(μ_a) - log(y!))

    All inputs are arrays of length n (number of matches).
    log_fact_home and log_fact_away are precomputed.
    """
    log_mu_home = np.log(mu_home + 1e-15)
    log_mu_away = np.log(mu_away + 1e-15)

    log_p_home = -mu_home + home_scores * log_mu_home - log_fact_home
    log_p_away = -mu_away + away_scores * log_mu_away - log_fact_away

    return log_p_home + log_p_away

In [27]:
def ip_grid(mu_home, mu_away):
    """
    Compute prediction probability grid for independent Poisson.

    P(X=x, Y=y) = P(X=x) × P(Y=y)

    Uses precomputed PRED_X_FLAT, PRED_Y_FLAT, PRED_LOG_FACT_X, PRED_LOG_FACT_Y.
    Returns (MAX_GOALS_PRED+1) × (MAX_GOALS_PRED+1) probability matrix.
    """
    log_mu_home = np.log(mu_home + 1e-15)
    log_mu_away = np.log(mu_away + 1e-15)

    log_p_home = -mu_home + PRED_X_FLAT * log_mu_home - PRED_LOG_FACT_X
    log_p_away = -mu_away + PRED_Y_FLAT * log_mu_away - PRED_LOG_FACT_Y

    log_probs = log_p_home + log_p_away
    return np.exp(log_probs).reshape(PRED_SHAPE)

In [28]:
def fit_from_cache(train_cache, mi_weights, warm_start=None):
    """
    Fit independent Poisson using precomputed cache.

    Parameters:
        params[0] = constant (log baseline scoring rate)
        params[1] = home advantage
        params[2:2+K] = team abilities

    μ_home = exp(constant + home_effect + ability_diff)
    μ_away = exp(constant - home_effect - ability_diff)

    No λ₃ parameter — goals are independent.
    """
    K = train_cache["K"]
    h_indices = train_cache["h_indices"]
    a_indices = train_cache["a_indices"]
    home_scores = train_cache["home_scores"]
    away_scores = train_cache["away_scores"]
    not_neutral = train_cache["not_neutral"]
    w_time = train_cache["w_time"]
    categories = train_cache["categories"]
    log_fact_home = train_cache["log_fact_home"]
    log_fact_away = train_cache["log_fact_away"]

    # Unbalanced: temporal × importance only
    w_importance = np.array([mi_weights[c] for c in categories])
    w = w_time * w_importance

    def neg_log_lik(params):
        constant = params[0]
        home_adv = params[1]
        abilities = params[2:2 + K]

        h_effect = np.where(not_neutral, home_adv, 0.0)
        ab_diff = abilities[h_indices] - abilities[a_indices]

        mu_home = np.exp(np.clip(constant + h_effect + ab_diff, -10, 5))
        mu_away = np.exp(np.clip(constant - h_effect - ab_diff, -10, 5))

        log_probs = ip_logpmf(
            mu_home, mu_away,
            home_scores, away_scores,
            log_fact_home, log_fact_away,
        )
        return -np.sum(w * log_probs)

    # 2 global params + K abilities (no log_lam3)
    if warm_start is not None and len(warm_start) == 2 + K:
        params0 = warm_start.copy()
    else:
        params0 = np.zeros(2 + K)
        params0[0] = train_cache["log_mean_goals"]
        params0[1] = 0.1

    result = minimize(
        neg_log_lik,
        params0,
        method="L-BFGS-B",
        options={"maxiter": 3000, "ftol": 1e-9},
    )

    params = result.x
    abilities = params[2:2 + K]
    abilities = abilities - abilities.mean()

    return {
        "constant": params[0],
        "home": params[1],
        "abilities_arr": abilities,
        "raw_params": params,
    }


def evaluate_from_cache(model, test_cache):
    """Evaluate model on test set — returns (sum_rps, count)."""
    n_valid = test_cache["n_valid"]
    if n_valid == 0:
        return 0.0, 0

    h_indices = test_cache["h_indices"]
    a_indices = test_cache["a_indices"]
    not_neutral = test_cache["not_neutral"]
    cum_actual = test_cache["cum_actual"]

    constant = model["constant"]
    home_adv = model["home"]
    abilities = model["abilities_arr"]

    h_effect = np.where(not_neutral, home_adv, 0.0)
    ab_diff = abilities[h_indices] - abilities[a_indices]

    mu_home = np.exp(constant + h_effect + ab_diff)
    mu_away = np.exp(constant - h_effect - ab_diff)

    sum_rps = 0.0
    for i in range(n_valid):
        prob_grid = ip_grid(mu_home[i], mu_away[i])

        p_hw = prob_grid[MASK_HOME_WIN].sum()
        p_d = prob_grid[MASK_DRAW].sum()
        p_aw = prob_grid[MASK_AWAY_WIN].sum()

        total = p_hw + p_d + p_aw
        if total > 0:
            p_hw /= total
            p_d /= total
            p_aw /= total

        cum_pred_0 = p_hw
        cum_pred_1 = p_hw + p_d

        sum_rps += 0.5 * (
            (cum_pred_0 - cum_actual[i, 0]) ** 2
            + (cum_pred_1 - cum_actual[i, 1]) ** 2
        )

    return sum_rps, n_valid


### 7. Evaluate Paper's Baseline

In [29]:
def evaluate_weights(window_caches, mi_weights, warm_starts=None, verbose=False):
    """Run a weight combination across all windows with warm starting."""
    total_rps = 0.0
    total_matches = 0
    new_warm_starts = {}
    failed_windows = 0

    for w_idx, (train_cache, test_cache, info) in enumerate(window_caches):
        try:
            ws = warm_starts.get(w_idx) if warm_starts else None
            model = fit_from_cache(train_cache, mi_weights, warm_start=ws)
            new_warm_starts[w_idx] = model["raw_params"]

            s_rps, n = evaluate_from_cache(model, test_cache)
            total_rps += s_rps
            total_matches += n

            if verbose and (w_idx + 1) % 10 == 0:
                running_avg = total_rps / total_matches if total_matches > 0 else 0
                print(f"      Window {w_idx+1}/{len(window_caches)} | "
                      f"running avg RPS: {running_avg:.6f} | "
                      f"matches so far: {total_matches}")
        except Exception as e:
            failed_windows += 1
            if verbose:
                print(f"      Window {w_idx+1} FAILED: {e}")
            continue

    if verbose and failed_windows > 0:
        print(f"       {failed_windows} windows failed")

    avg_rps = total_rps / total_matches if total_matches > 0 else np.nan
    return avg_rps, total_matches, new_warm_starts

In [30]:
print("-" * 60)
print("INDEPENDENT POISSON MODEL")
print("-" * 60)
print("Evaluating paper's baseline weights...")
paper_weights = [1.0, 2.5, 3.0, 4.0]
paper_rps, paper_n, _ = evaluate_weights(window_caches, paper_weights, verbose=True)
print(f"\nPaper's weights:     [friendly=1.0, qualifier=2.5, confederation=3.0, worldcup=4.0]")
print(f"Paper's Avg RPS:     {paper_rps:.6f}")
print(f"Matches evaluated:   {paper_n}")

------------------------------------------------------------
INDEPENDENT POISSON MODEL
------------------------------------------------------------
Evaluating paper's baseline weights...
      Window 10/73 | running avg RPS: 0.191606 | matches so far: 679
      Window 20/73 | running avg RPS: 0.202628 | matches so far: 1116
      Window 30/73 | running avg RPS: 0.192657 | matches so far: 1973
      Window 40/73 | running avg RPS: 0.197751 | matches so far: 2468
      Window 50/73 | running avg RPS: 0.198804 | matches so far: 3147
      Window 60/73 | running avg RPS: 0.199143 | matches so far: 3783
      Window 70/73 | running avg RPS: 0.198858 | matches so far: 4427

Paper's weights:     [friendly=1.0, qualifier=2.5, confederation=3.0, worldcup=4.0]
Paper's Avg RPS:     0.196168
Matches evaluated:   4768


### 8. Grid Search

In [31]:
friendly_vals      = [0.75, 1.00, 1.25, 1.50, 1.75]
qualifier_vals     = [1.50, 1.75, 2.00, 2.25, 2.50, 2.75]
confederation_vals = [1.50, 1.75, 2.00, 2.25, 2.50, 2.75, 3.00]
worldcup_vals      = [3.5, 4.0, 4.25, 4.5, 5.0]

grid = list(itertools.product(
    friendly_vals,
    qualifier_vals,
    confederation_vals,
    worldcup_vals,
))

In [32]:
total_combos = len(grid)
total_windows = len(window_caches)
total_fits = total_combos * total_windows

In [33]:
print(f"\nGrid search configuration:")
print(f"  Friendly values:      {friendly_vals}")
print(f"  Qualifier values:     {qualifier_vals}")
print(f"  Confederation values: {confederation_vals}")
print(f"  World Cup values:     {worldcup_vals}")
print(f"  ─────────────────────────────────")
print(f"  Weight combinations:  {total_combos}")
print(f"  Rolling windows:      {total_windows}")
print(f"  Total model fits:     {total_fits}")


Grid search configuration:
  Friendly values:      [0.75, 1.0, 1.25, 1.5, 1.75]
  Qualifier values:     [1.5, 1.75, 2.0, 2.25, 2.5, 2.75]
  Confederation values: [1.5, 1.75, 2.0, 2.25, 2.5, 2.75, 3.0]
  World Cup values:     [3.5, 4.0, 4.25, 4.5, 5.0]
  ─────────────────────────────────
  Weight combinations:  1050
  Rolling windows:      73
  Total model fits:     76650


In [34]:
results = []
start_time = time.time()
prev_warm_starts = None

best_rps_so_far = np.inf
best_weights_so_far = None
completed = 0
failed_count = 0

In [35]:
print("\n" + "-" * 90)
print("GRID SEARCH STARTED — INDEPENDENT POISSON (CLEAN MAPPING + UNBALANCED)")
print("-" * 90)
print(f"Start time: {time.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Paper's baseline: RPS = {paper_rps:.6f}")
print(f"{'─' * 90}")


------------------------------------------------------------------------------------------
GRID SEARCH STARTED — INDEPENDENT POISSON (CLEAN MAPPING + UNBALANCED)
------------------------------------------------------------------------------------------
Start time: 2026-03-17 02:20:39
Paper's baseline: RPS = 0.196168
──────────────────────────────────────────────────────────────────────────────────────────


In [36]:
for idx, weights in enumerate(grid):
    combo_start = time.time()

    avg_rps, n_matches, prev_warm_starts = evaluate_weights(
        window_caches, list(weights), warm_starts=prev_warm_starts
    )

    combo_time = time.time() - combo_start
    completed += 1

    if np.isnan(avg_rps):
        failed_count += 1

    results.append({
        "friendly": weights[0],
        "qualifier": weights[1],
        "confederation": weights[2],
        "worldcup": weights[3],
        "avg_rps": avg_rps,
        "n_matches": n_matches,
    })

    # Track best
    new_best = False
    if not np.isnan(avg_rps) and avg_rps < best_rps_so_far:
        best_rps_so_far = avg_rps
        best_weights_so_far = weights
        new_best = True

    # Progress
    elapsed = time.time() - start_time
    avg_per_combo = elapsed / completed
    remaining = avg_per_combo * (total_combos - completed)
    pct = completed / total_combos * 100
    eta = time.strftime('%H:%M:%S', time.gmtime(remaining))

    # Log every 20 combos, on new best, or at the end
    if completed % 20 == 0 or new_best or completed == total_combos:
        status = "★ NEW BEST" if new_best else ""
        print(
            f"  [{completed:>4d}/{total_combos}] {pct:5.1f}% | "
            f"weights=[{weights[0]:.2f}, {weights[1]:.2f}, {weights[2]:.2f}, {weights[3]:.2f}] | "
            f"RPS={avg_rps:.6f} | "
            f"best={best_rps_so_far:.6f} | "
            f"this={combo_time:.1f}s | "
            f"elapsed={elapsed/60:.1f}min | "
            f"ETA={eta} {status}"
        )

total_time = time.time() - start_time



  [   1/1050]   0.1% | weights=[0.75, 1.50, 1.50, 3.50] | RPS=0.195427 | best=0.195427 | this=111.3s | elapsed=1.9min | ETA=08:26:35 ★ NEW BEST
  [  20/1050]   1.9% | weights=[0.75, 1.50, 2.25, 5.00] | RPS=0.196215 | best=0.195427 | this=21.8s | elapsed=13.8min | ETA=11:52:36 
  [  40/1050]   3.8% | weights=[0.75, 1.75, 1.50, 5.00] | RPS=0.196061 | best=0.195427 | this=20.8s | elapsed=24.7min | ETA=10:24:20 
  [  60/1050]   5.7% | weights=[0.75, 1.75, 2.50, 5.00] | RPS=0.196437 | best=0.195427 | this=19.1s | elapsed=34.8min | ETA=09:33:37 
  [  80/1050]   7.6% | weights=[0.75, 2.00, 1.75, 5.00] | RPS=0.196336 | best=0.195427 | this=18.9s | elapsed=45.7min | ETA=09:13:35 
  [ 100/1050]   9.5% | weights=[0.75, 2.00, 2.75, 5.00] | RPS=0.196670 | best=0.195427 | this=20.7s | elapsed=55.5min | ETA=08:47:39 
  [ 120/1050]  11.4% | weights=[0.75, 2.25, 2.00, 5.00] | RPS=0.196605 | best=0.195427 | this=19.7s | elapsed=66.7min | ETA=08:37:02 
  [ 140/1050]  13.3% | weights=[0.75, 2.25, 3.00, 5.

In [37]:

print(f"\n{'-' * 90}")
print(f"GRID SEARCH COMPLETE — INDEPENDENT POISSON")
print(f"{'-' * 90}")
print(f"  End time:        {time.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"  Total time:      {total_time/60:.1f} min ({total_time/3600:.2f} hours)")
print(f"  Combos evaluated: {completed}")
print(f"  Combos failed:   {failed_count}")
print(f"  Avg per combo:   {total_time/completed:.2f}s")
print(f"  Best RPS:        {best_rps_so_far:.6f}")
print(f"  Best weights:    [{best_weights_so_far[0]:.2f}, {best_weights_so_far[1]:.2f}, "
      f"{best_weights_so_far[2]:.2f}, {best_weights_so_far[3]:.2f}]")


------------------------------------------------------------------------------------------
GRID SEARCH COMPLETE — INDEPENDENT POISSON
------------------------------------------------------------------------------------------
  End time:        2026-03-17 11:25:58
  Total time:      545.3 min (9.09 hours)
  Combos evaluated: 1050
  Combos failed:   0
  Avg per combo:   31.16s
  Best RPS:        0.195324
  Best weights:    [1.50, 1.75, 1.50, 3.50]


### Results

In [38]:
results_df = pd.DataFrame(results).dropna(subset=["avg_rps"])
results_df = results_df.sort_values("avg_rps").reset_index(drop=True)

In [60]:
results_df.to_csv("grid_search_independent_poisson_results.csv", index=False)
print("Full results saved to grid_search_independent_poisson_results.csv\n")

Full results saved to grid_search_independent_poisson_results.csv



In [40]:
print("-" * 95)
print("TOP 10 WEIGHT COMBINATIONS — INDEPENDENT POISSON (lowest RPS = best)")
print("-" * 95)
print(f"{'Rank':<6} {'Friendly':<10} {'Qualifier':<10} {'Confederation':<14} {'WorldCup':<10} {'Avg RPS':<12} {'Matches':<8}")
print("-" * 80)
for i, row in results_df.head(10).iterrows():
    print(f"{i+1:<6} {row['friendly']:<10.2f} {row['qualifier']:<10.2f} "
          f"{row['confederation']:<14.2f} {row['worldcup']:<10.2f} "
          f"{row['avg_rps']:<12.6f} {int(row['n_matches']):<8}")


-----------------------------------------------------------------------------------------------
TOP 10 WEIGHT COMBINATIONS — INDEPENDENT POISSON (lowest RPS = best)
-----------------------------------------------------------------------------------------------
Rank   Friendly   Qualifier  Confederation  WorldCup   Avg RPS      Matches 
--------------------------------------------------------------------------------
1      1.50       1.75       1.50           3.50       0.195324     4768    
2      1.75       2.00       1.50           5.00       0.195340     4768    
3      1.75       2.00       1.75           5.00       0.195340     4768    
4      1.50       2.00       1.50           3.50       0.195341     4768    
5      1.75       2.00       1.50           4.50       0.195343     4768    
6      1.75       2.00       1.75           4.50       0.195343     4768    
7      1.50       1.75       1.50           4.25       0.195344     4768    
8      1.75       2.00       1.50         

In [41]:
print("\nBOTTOM 10 (worst combinations):")
print(f"{'Rank':<6} {'Friendly':<10} {'Qualifier':<10} {'Confederation':<14} {'WorldCup':<10} {'Avg RPS':<12}")
print("-" * 72)
for i, row in results_df.tail(10).iterrows():
    print(f"{i+1:<6} {row['friendly']:<10.2f} {row['qualifier']:<10.2f} "
          f"{row['confederation']:<14.2f} {row['worldcup']:<10.2f} "
          f"{row['avg_rps']:<12.6f}")


BOTTOM 10 (worst combinations):
Rank   Friendly   Qualifier  Confederation  WorldCup   Avg RPS     
------------------------------------------------------------------------
1041   0.75       2.75       2.75           3.50       0.197142    
1042   0.75       2.75       2.75           4.00       0.197156    
1043   0.75       2.75       2.75           4.25       0.197163    
1044   0.75       2.75       2.75           4.50       0.197168    
1045   0.75       2.75       2.75           5.00       0.197183    
1046   0.75       2.75       3.00           3.50       0.197214    
1047   0.75       2.75       3.00           4.00       0.197227    
1048   0.75       2.75       3.00           4.25       0.197235    
1049   0.75       2.75       3.00           4.50       0.197241    
1050   0.75       2.75       3.00           5.00       0.197256    


In [58]:
best = results_df.iloc[0]


print("FINAL COMPARISON INDEPENDENT POISSON")


print(f"\n  Paper's weights:")
print(f"    Friendly:       1.00")
print(f"    Qualifier:      2.50")
print(f"    Confederation:  3.00")
print(f"    World Cup:      4.00")
print(f"    Avg RPS:        {paper_rps:.6f}")

print(f"\n  Best weights found (clean mapping, unbalanced):")
print(f"    Friendly:       {best['friendly']:.2f}")
print(f"    Qualifier:      {best['qualifier']:.2f}")
print(f"    Confederation:  {best['confederation']:.2f}")
print(f"    World Cup:      {best['worldcup']:.2f}")
print(f"    Avg RPS:        {best['avg_rps']:.6f}")

FINAL COMPARISON INDEPENDENT POISSON

  Paper's weights:
    Friendly:       1.00
    Qualifier:      2.50
    Confederation:  3.00
    World Cup:      4.00
    Avg RPS:        0.196168

  Best weights found (clean mapping, unbalanced):
    Friendly:       1.50
    Qualifier:      1.75
    Confederation:  1.50
    World Cup:      3.50
    Avg RPS:        0.195324


In [59]:
improvement_vs_paper = ((paper_rps - best["avg_rps"]) / paper_rps) * 100
print(f"New weights vs Paper's weights: {improvement_vs_paper:+.4f}%")


if improvement_vs_paper > 0:
    print("New weights are BETTER than the paper's!")
elif improvement_vs_paper < 0:
    print("The paper's weights perform better.")
else:
    print("  = Same performance.")

New weights vs Paper's weights: +0.4299%
New weights are BETTER than the paper's!
